#### 1. Prepare the modules

In [ ]:
!pip install pycuda
!pip install rasterio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 9.1 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2025.1.2-cp312-cp312-linux_x86_64.whl size=659050 sha256=45b3d4101239e6e3ea34cdf21d9880db6ca327e895c12d7143090ba862f833fc
  Stored in directory: /root/.cache/pip/wheels/d5/36/f3/ac5f09d768cad3fa15d5a3449bdfe65c3de58e69d036c73228
Successfully built pycuda
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 115.1 MB/s eta 0:00:00


In [ ]:
from pycuda.compiler import SourceModule
import pycuda
from pycuda import gpuarray
from pycuda import compiler
import pycuda.autoinit             # PyCuda autoinit
import pycuda.driver as cuda       # PyCuda In, Out helpers
import os, os.path
import rasterio as rio
from osgeo import gdal
from osgeo.gdalconst import *
import numpy as np
import rasterio
import time
import math
# Load the Drive helper and mount
from google.colab import drive
import matplotlib.cm as colormap   # Library to plot
import numpy                       # Fast math library
import matplotlib.image as mpimg       # reading images to numpy arrays
import matplotlib.pyplot as plt        # to plot any graph
import scipy.ndimage as ndi            # to determine shape centrality
# matplotlib setup

%matplotlib inline
from pylab import rcParams
rcParams['figure.figsize'] = (8, 8)      # setting default size of plots

print("pycuda installed and ready to use.")

print("%d device(s) found." % cuda.Device.count())
for ordinal in range(cuda.Device.count()):
    dev = cuda.Device(ordinal)
    print ("Device #%d: %s" % (ordinal, dev.name()))
print (cuda)

pycuda installed and ready to use.
1 device(s) found.
Device #0: Tesla T4
<module 'pycuda.driver' from '/usr/local/lib/python3.12/dist-packages/pycuda/driver.py'>


In [ ]:
# This will prompt for authorization.
drive.mount('/content/drive')

# replace "deeplearn-city/treepedia-workshop" by your own drive path
dsm_file = "/content/drive/MyDrive/Colab Notebooks/data/row11-col17.tif"
os.path.exists(dsm_file)

dsm_dataset = rio.open(dsm_file)

dsm_bounds = dsm_dataset.bounds
dsm_img = dsm_dataset.read(1)
print('The dsm bound is:', dsm_bounds)


transform = dsm_dataset.transform

# The pixel width and height (scale) can be found as follows:
pixel_width = transform[0]  # Scale in the x-direction
pixel_height = -transform[4]  # Scale in the y-direction (usually negative)

# You can calculate a unified scale if needed (average or reciprocal)
scale = 1 / pixel_width  # This will give you a scale factor


Mounted at /content/drive
The dsm bound is: BoundingBox(left=2712748.0, bottom=259294.0, right=2717948.0, top=264494.0)


### 2.

In [ ]:
# Define the CUDA kernel for the shadow calculation
kernel_shadow = """
#define PI 3.1415926

__global__ void calculate_shadow(float *dsm, bool *shadow, int width, int height, float cell_size, float sun_azimuth, float sun_elevation) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    // Check if the thread is within the bounds of the image
    if (x >= width || y >= height)
        return;

    int idx = y * width + x;

    // Convert sun azimuth and elevation to radians
    float azimuth_rad = sun_azimuth * 3.14159 / 180.0;
    float elevation_rad = sun_elevation * 3.14159 / 180.0;

    // Direction of the shadow ray based on sun azimuth
    float dx = sin(azimuth_rad);
    float dy = -cos(azimuth_rad);
    float dz = tan(elevation_rad);

    // Initial height at the current pixel
    float initial_height = dsm[idx];
    bool in_shadow = false;

    // Trace the shadow ray
    for (float t = cell_size; t < 2000.0f; t += cell_size) { // Limit tracing distance
        int x_offset = x + int(dx * t / cell_size);
        int y_offset = y + int(dy * t / cell_size);

        if (x_offset < 0 || x_offset >= width || y_offset < 0 || y_offset >= height)
            break;  // Out of bounds

        int offset_idx = y_offset * width + x_offset;

        // Calculate height along the ray
        float height_along_ray = initial_height + dz * t;

        // Check if there is a higher point along the path
        if (dsm[offset_idx] > height_along_ray) {
            in_shadow = true;
            break;
        }
    }

    // Store shadow result: 1 for shadow, 0 for no shadow
    shadow[idx] = in_shadow;
}
"""

# Compile the kernel
mod = SourceModule(kernel_shadow)
calculate_shadows = mod.get_function("calculate_shadow")

In [ ]:
# Prepare parameters and allocate memory
H, W = dsm_img.shape

d_dsm = cuda.mem_alloc(dsm_img.nbytes)
cuda.memcpy_htod(d_dsm, dsm_img)

d_shadow    = cuda.mem_alloc(dsm_img.size)
host_shadow = np.empty((H, W), dtype=np.bool_)

block = (16, 16, 1)
grid  = (int(np.ceil(W / block[0])), int(np.ceil(H / block[1])))

# Define a function
def run_shadow(azimuth_deg: float, elevation_deg: float) -> np.ndarray:
    calculate_shadows(
        d_dsm, d_shadow,
        np.int32(W), np.int32(H),
        np.float32(cell_size),
        np.float32(azimuth_deg), np.float32(elevation_deg),
        block=block, grid=grid
    )
    cuda.memcpy_dtoh(host_shadow, d_shadow)
    return host_shadow.copy()

In [ ]:
# solar positions from noaa (Oct 25th 2025, Philly; from 8am to 4pm)
solarpos = [
    (8, 122.28, 16.34),
    (9, 134.68, 25.31),
    (10, 149.35, 32.38),
    (11, 166.37, 36.71),
    (12, 184.75, 37.59),
    (13, 202.61, 34.85),
    (14, 218.45, 28.98),
    (15, 231.87, 20.82),
    (16, 243.29, 11.14),
]
assert len(solarpos) == 9

cell_size = 1.0

In [ ]:
from pathlib import Path
# create loop to calculate the shadow for 9 times
out_dir = Path("/content/drive/MyDrive/Colab Notebooks/data")   # output folder
out_dir.mkdir(exist_ok=True, parents=True)

for hour, az, el in solarpos:           # loop
    print(f"Computing {hour}:00  az={az:.1f}°, el={el:.1f}° ...")
    shadow_bool = run_shadow(az, el)

    # Save PNG
    png_path = out_dir / f"shadow_{hour:02d}.png"
    plt.imsave(png_path,
           shadow_bool.astype(np.uint8),
           cmap="plasma_r", vmin=0, vmax=1)
    print("saved:", png_path)

print(f"\nFinish：9 PNG have been saved：{out_dir.resolve()}")


Computing 8:00  az=122.3°, el=16.3° ...
saved: /content/drive/MyDrive/Colab Notebooks/data/shadow_08.png
Computing 9:00  az=134.7°, el=25.3° ...
saved: /content/drive/MyDrive/Colab Notebooks/data/shadow_09.png
Computing 10:00  az=149.3°, el=32.4° ...
saved: /content/drive/MyDrive/Colab Notebooks/data/shadow_10.png
Computing 11:00  az=166.4°, el=36.7° ...
saved: /content/drive/MyDrive/Colab Notebooks/data/shadow_11.png
Computing 12:00  az=184.8°, el=37.6° ...
saved: /content/drive/MyDrive/Colab Notebooks/data/shadow_12.png
Computing 13:00  az=202.6°, el=34.9° ...
saved: /content/drive/MyDrive/Colab Notebooks/data/shadow_13.png
Computing 14:00  az=218.4°, el=29.0° ...
saved: /content/drive/MyDrive/Colab Notebooks/data/shadow_14.png
Computing 15:00  az=231.9°, el=20.8° ...
saved: /content/drive/MyDrive/Colab Notebooks/data/shadow_15.png
Computing 16:00  az=243.3°, el=11.1° ...
saved: /content/drive/MyDrive/Colab Notebooks/data/shadow_16.png

Finish：9 PNG have been saved：/content/drive/MyD

In [ ]:
## Make GIF
import imageio.v2 as imageio

png_paths = sorted(out_dir.glob("shadow_*.png"))
frames = [imageio.imread(p) for p in png_paths]
gif_path = out_dir / "shadow_day.gif"
imageio.mimsave(gif_path, frames, duration=1.5, loop=0)
print("Saved GIF:", gif_path)


Saved GIF: /content/drive/MyDrive/Colab Notebooks/data/shadow_day.gif
